# Module 2 Cleanup: Snowflake Postgres Portal

This notebook removes all objects created by Module 2 (`hol-module2.ipynb`) and restores the EPOWER Agent to its Module 1 state.

**Execution order matters:**

| Step | Where | What |
|------|-------|------|
| 1 | **Postgres** (psql) | Drop tables, pipelines, extensions |
| 2 | **Snowflake** (this notebook) | Drop Iceberg table, views, catalog integration, network objects, Postgres instance |

> **Why this order?** The Postgres instance must be cleaned up *before* it is dropped from Snowflake. Once dropped, you lose access to the Postgres database and any data left behind is orphaned.

## Step 1: Postgres Cleanup (run in psql)

Run this in your Postgres client **before** continuing:

```bash
psql service=my_epower_portal -f cleanup-module2-postgres.sql
```

Or manually:

```sql
SELECT incremental.drop_pipeline('sync_portal_activity_to_iceberg');
DROP TABLE IF EXISTS portal_activity_log_iceberg;
DROP TABLE IF EXISTS portal_activity_log CASCADE;
DROP TABLE IF EXISTS service_requests CASCADE;
DROP TABLE IF EXISTS tariff_orders CASCADE;
DROP TABLE IF EXISTS meter_readings CASCADE;
DROP TABLE IF EXISTS portal_users CASCADE;
DROP EXTENSION IF EXISTS pg_incremental CASCADE;
DROP EXTENSION IF EXISTS pg_cron CASCADE;
DROP EXTENSION IF EXISTS pg_lake CASCADE;
```

Once done, continue with Step 2 below.

## Step 2: Snowflake Cleanup

Run the following cells to remove all Snowflake objects created by Module 2.

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

DROP ICEBERG TABLE IF EXISTS EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;
DROP VIEW IF EXISTS EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT;
DROP SEMANTIC VIEW IF EXISTS EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW;
DROP STAGE IF EXISTS EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEED_STAGE;
DROP CATALOG INTEGRATION IF EXISTS PORTAL_POSTGRES_CATALOG;
DROP NETWORK POLICY IF EXISTS EPOWER_PG_POLICY;
DROP NETWORK RULE IF EXISTS EPOWER_PG_INGRESS;

In [ ]:
%%sql
-- This is irreversible — the Postgres instance and all its data will be deleted
DROP POSTGRES INSTANCE IF EXISTS MY_EPOWER_PORTAL;

## Step 3: Restore Agent to Module 1 State

The Module 2 notebook added `portal_analyst` to the EPOWER Agent. This cell recreates the agent **without** the portal tool.

In [ ]:
%%sql
USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;

CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: You MUST always respond in the SAME language as the user's question.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions → energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products → customer_energy_analyst
    - Sales/contracts → energy_sales_analyst
    - Billing → billing_analyst
    - Service tickets → service_analyst
    - HR data → hr_analyst
    - Electricity market prices, day-ahead → epulse_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export → vpp_telemetry_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, grid import/export"}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

## Verification

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

SHOW POSTGRES INSTANCES LIKE 'MY_EPOWER%';
SHOW CATALOG INTEGRATIONS LIKE 'PORTAL%';
SHOW NETWORK POLICIES LIKE 'EPOWER_PG%';

---

Module 2 cleanup complete. The EPOWER demo is back to Module 1 state.

Don't forget to remove the psql connection:

```bash
# Remove from ~/.pg_service.conf: [my_epower_portal] section
# Remove from ~/.pgpass: the corresponding line
```